In [ ]:
import numpy as np
import pandas as pd

import diffuse_w_background as dfb
import focus
import os
import cv2

from matplotlib import pyplot as plt
from tqdm import tqdm

from diffuse_w_background import img_dir, mask_dir

OVIS_PATH = ""
PATH_TO_IRUO = ""

In [ ]:
testim = dfb.get_img('giant_panda_o0_n1.jpg', img_dir=img_dir)[...,::-1]
testmask = dfb.get_mask('giant_panda_o0_n1.jpg', mask_dir=mask_dir)
testmask_b = dfb.quick_background_mask(testmask)

imb = [testim, testmask, testmask_b]

def write(im, msk, bmsk, sv):
    cv2.imwrite(sv, im)#[...,::-1])
    cv2.imwrite(sv.replace('imgs', 'masks'), msk)
    cv2.imwrite(sv.replace('imgs', 'bmsks'), bmsk)
    
def test_output(func):
    return func(testim, testmask, testmask_b)

def display_output(rm, flip=False):
    fig, ax = plt.subplots(1,2, figsize=[1.5,1])
    for i, (a, r) in enumerate(zip(ax, rm)):
        if flip and i==0: a.imshow(r[:,:,::-1]); a.axis('off')
        else: a.imshow(r); a.axis('off')
    fig.suptitle('occlusion: %.3f/%.3f' % dfb.calculate_occlusion_w_mask2(testmask, rm[1], testmask_b, rm[-1]), size='x-small', y=0.8)
    fig.tight_layout()

In [ ]:
rm = test_output(dfb.random_mask)
display_output(rm)

In [ ]:
rm = dfb.rotated_gate_occlusion3(*imb, widths=(10,4), spacings=(20,8))
display_output(rm)

In [ ]:
def rand_gen_rgates(im, mask, maskb):
    occs = (1,1)
    while (occs[0] < 0.05) | (occs[0] > 0.95):
        w, s = np.random.randint(1,20), np.random.randint(1,20)
        wb, sb = int(w/2)+np.random.randint(-4,4), s
        if wb <= 1: wb += np.random.randint(0,4)
        if wb/sb < 0.05: wb += np.random.randint(1,np.amax([int(sb/2),2]))
        if (w>=s) | (wb>=sb) | (wb/sb >= 0.5): continue
        randout = dfb.rotated_gate_occlusion3(im, mask, maskb, angle=np.random.randint(-180,180), widths=(w,wb), spacings=(s,sb))
        if np.random.random() > 0.5: 
            randout = dfb.rotated_gate_occlusion3(*randout, angle=np.random.randint(-180,180), widths=(w,wb), spacings=(s,sb))
        occs = dfb.calculate_occlusion_w_mask2(mask, randout[1], maskb, randout[-1])
    return randout
        
rm = rand_gen_rgates(*imb)
display_output(rm)

In [ ]:
def rand_gen_rights(im, mask, maskb):
    occs = (1,1)
    while (occs[0] < 0.05) | (occs[0] > 0.95) | (occs[1] > 0.6):
        w, s = np.random.randint(1,20), np.random.randint(1,20)
        wb, sb = int(w/2)+np.random.randint(-4,4), s
        if wb <= 1: wb += np.random.randint(0,4)
        if wb/sb < 0.05: wb += np.random.randint(1,np.amax([int(sb/2),2]))
        if (w>=s) | (wb>=sb) | (wb/sb >= 0.5): continue
        randout = dfb.xhatch_occlusion3(im, mask, maskb, widths=(w,wb), spacings=(s,sb))
        occs = dfb.calculate_occlusion_w_mask2(mask, randout[1], maskb, randout[-1])
    return randout

rm = rand_gen_rights(*imb)
display_output(rm)

In [ ]:
def rand_gen_box(im, mask, maskb):
    occs = (1,1)
    while (occs[0] < 0.05) | (occs[0] > 0.95):
        randout = dfb.random_mask(im, mask, maskb, mode='gray')
        occs = dfb.calculate_occlusion_w_mask2(mask, randout[1], maskb, randout[-1])
    return randout

rm = rand_gen_box(*imb)
display_output(rm)

In [ ]:
def rand_gen_gates(im, mask, maskb, mode='vertical'):
    occs = (1,1)
    while (occs[0] < 0.05) | (occs[0] > 0.95) | (occs[1] > 0.6):
        w, s = np.random.randint(1,20), np.random.randint(1,20)
        wb, sb = int(w/2)+np.random.randint(-4,4), s
        if wb <= 1: wb += np.random.randint(0,4)
        if wb/sb < 0.05: wb += np.random.randint(1,np.amax([int(sb/2),2]))
        if (w>=s) | (wb>=sb) | (wb/sb >= 0.5): continue
        randout = dfb.gate_occlusion3(im, mask, maskb, widths=(w,wb), spacings=(s,sb), mode=mode)
        occs = dfb.calculate_occlusion_w_mask2(mask, randout[1], maskb, randout[-1])
    return randout

rm = rand_gen_gates(*imb)
display_output(rm)

In [ ]:
clear_0 = focus.annos(0)
focus.focus(clear_0)
clear_0 = focus.filter_clear(clear_0)
clear_0.drop(columns=['px', 'blur']).to_csv('clear_0.csv', sep=' ', header=False, index=False)

In [ ]:
def exists_or_make(x): 
    if not os.path.exists(x): os.mkdir(x)

exists_or_make('gen_imgs')
exists_or_make('gen_masks')
exists_or_make('gen_bmsks')

for x in [os.path.join('gen_imgs', x) for x in ['vgates', 'hgates', 'xgates', 'rotgates', 'boxes']]: 
    exists_or_make(x)

for x in [os.path.join('gen_masks', x) for x in ['vgates', 'hgates', 'xgates', 'rotgates', 'boxes']]: 
    exists_or_make(x)

for x in [os.path.join('gen_bmsks', x) for x in ['vgates', 'hgates', 'xgates', 'rotgates', 'boxes']]: 
    exists_or_make(x)

In [ ]:
box_anno = []
vga_anno = []
hga_anno = []
xga_anno = []
rga_anno = []
lea_anno = []

for rec in tqdm(clear_0.iloc, desc='Generating images', total=len(clear_0)):
    f = rec.fname
    c = rec.cls

    try:
        imin = dfb.get_img(focus.floc(f))
        if os.path.exists(focus.mloc(f)):
            mskin = dfb.get_mask(focus.mloc(f),mask_dir=f'{PATH_TO_IRUO}/seg-masks')
            bmskin = dfb.quick_background_mask(mskin)
        else: continue
    except:
        continue

    if np.sum(mskin) < 1: continue

    imb0 = (imin, mskin, bmskin)

    # boxes
    randout = rand_gen_box(*imb0)
    occ = dfb.calculate_occlusion_w_mask2(mskin, randout[1], bmskin, randout[-1])
    sv = 'gen_imgs/boxes/o%i_b%i_%s' % (*[int(100*o) for o in occ], f.split(os.sep)[-1])
    box_anno.append({'fname': sv, 'cls': c})
    write(*randout, sv)

    # vgates
    randout = rand_gen_gates(*imb0, mode='vertical')
    occ = dfb.calculate_occlusion_w_mask2(mskin, randout[1], bmskin, randout[-1])
    sv = 'gen_imgs/vgates/o%i_b%i_%s' % (*[int(100*o) for o in occ], f.split(os.sep)[-1])
    vga_anno.append({'fname': sv, 'cls': c})
    write(*randout, sv)

    # hgates
    randout = rand_gen_gates(*imb0, mode='horizontal')
    occ = dfb.calculate_occlusion_w_mask2(mskin, randout[1], bmskin, randout[-1])
    sv = 'gen_imgs/hgates/o%i_b%i_%s' % (*[int(100*o) for o in occ], f.split(os.sep)[-1])
    hga_anno.append({'fname': sv, 'cls': c})
    write(*randout, sv)

    # xgates
    randout = rand_gen_rights(*imb0)
    occ = dfb.calculate_occlusion_w_mask2(mskin, randout[1], bmskin, randout[-1])
    sv = 'gen_imgs/xgates/o%i_b%i_%s' % (*[int(100*o) for o in occ], f.split(os.sep)[-1])
    xga_anno.append({'fname': sv, 'cls': c})
    write(*randout, sv)

    # rotgates
    randout = rand_gen_rgates(*imb0)
    occ = dfb.calculate_occlusion_w_mask2(mskin, randout[1], bmskin, randout[-1])
    sv = 'gen_imgs/rotgates/o%i_b%i_%s' % (*[int(100*o) for o in occ], f.split(os.sep)[-1])
    rga_anno.append({'fname': sv, 'cls': c})
    write(*randout, sv)

In [ ]:
exists_or_make('csvs')

for ana, afl in zip(['box', 'vga', 'hga', 'xga', 'rga'], [box_anno, vga_anno, hga_anno, xga_anno, rga_anno]):
    afl = pd.DataFrame.from_records(afl)
    afl.to_csv('csvs/%s.csv' % ana, sep=' ', header=False, index=False)

In [ ]:
anno_files = []
for anno_file in ['csvs/%s.csv' % a for a in ['box', 'hga', 'rga', 'vga', 'xga']]:
    if os.path.exists(anno_file):
        anno_files.append(pd.read_csv(anno_file, sep=' ', names=['fname', 'cls']))
if len(anno_files) > 0:
    anno_files = pd.concat(anno_files, axis=0)
    anno_files.to_csv('diffuse.csv', sep=' ', header=False, index=False)